#AI Risk Hotspots Early Warning System V2

In [51]:
# ==============================================================================
# main_analysis.py - V2 (Enhanced for Dynamic Forecasting)
# ==============================================================================

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import json
from google.colab import files


In [52]:
# --- 1. Load Cleaned Datasets ---
try:
    df_incidents = pd.read_csv('llm_classified_incidents.csv')
    df_compute = pd.read_csv('epoch_ai_compute_cleaned.csv')
    print("✅ Successfully loaded 'classified_incidents.csv' and 'epoch_ai_compute_cleaned.csv'.")
except FileNotFoundError as e:
    print(f"❌ ERROR: File not found. Please make sure '{e.filename}' is uploaded to this Colab session.")
    exit()


✅ Successfully loaded 'classified_incidents.csv' and 'epoch_ai_compute_cleaned.csv'.


In [53]:
# --- 2. Prepare Incident Data for Time-Series Analysis (Corrected) ---


# First, ensure the 'date' column is treated as a string.
df_incidents['date'] = df_incidents['date'].astype(str)
# Safely extract the first 10 characters (the YYYY-MM-DD part)
df_incidents['date_clean'] = df_incidents['date'].str.slice(0, 10)

# Now, convert this clean, consistent string to a proper datetime object.
df_incidents['date'] = pd.to_datetime(df_incidents['date_clean'], errors='coerce')


# Drop any rows where the date could still not be parsed (safer now)
df_incidents.dropna(subset=['date'], inplace=True)

# Create 'year' and 'quarter' columns
df_incidents['year'] = df_incidents['date'].dt.year
df_incidents['quarter_str'] = df_incidents['date'].dt.to_period('Q').astype(str)
print("-> Prepared incident data for time-series analysis with robust date parsing.")

-> Prepared incident data for time-series analysis with robust date parsing.


In [54]:
# ==============================================================================
# ### FIX (TASK 1) ###
# Exclude the last, potentially incomplete quarter from analysis that requires it.
# ==============================================================================
last_quarter_str = df_incidents['quarter_str'].max()
df_incidents_complete = df_incidents[df_incidents['quarter_str'] != last_quarter_str].copy()
print(f"-> For trend analysis, excluding the last partial quarter: {last_quarter_str}")

-> For trend analysis, excluding the last partial quarter: 2025Q4


#Task A: Generate "Risk Radar" Data (Incidents by Category Over Time)

In [55]:
print("\n--- Generating Risk Radar data ---")
# The Risk Radar should show ALL historical data, even the incomplete quarter.
risk_radar_data = df_incidents.groupby(['quarter_str', 'harm_category']).size().reset_index(name='incident_count')
risk_radar_data_json = risk_radar_data.to_dict(orient='records')
print("✅ 'Risk Radar' data is ready.")


--- Generating Risk Radar data ---
✅ 'Risk Radar' data is ready.


#Task B: Generate "Cause & Effect Correlator" Data

In [56]:
print("\n--- Generating Cause & Effect Correlator data ---")
incidents_by_year = df_incidents.groupby('year').size().reset_index(name='incident_count')
cause_effect_data = pd.merge(incidents_by_year, df_compute, on='year', how='inner')
cause_effect_data_json = cause_effect_data.to_dict(orient='records')
print("✅ 'Cause & Effect' data is ready.")


--- Generating Cause & Effect Correlator data ---
✅ 'Cause & Effect' data is ready.


#Task C: Generate "Hotspot Forecast" Data
##Generate forecasts for ALL categories for the interactive dashboard.

In [57]:
def generate_forecast_for_category(category_name, historical_data):
    """Generates a 3-year forecast for a specific harm category."""

    # Filter for the specific category
    category_historical = historical_data[historical_data['harm_category'] == category_name].copy()
    # Only use the last 3 years (12 quarters) of data to train the model for a more responsive forecast.
    category_historical = category_historical.tail(12)
    category_historical['type'] = 'Historical'

    # If there's not enough data to forecast, return just the historicals
    if len(category_historical) < 2:
        return category_historical.to_dict(orient='records')

    # Create a numeric time index for regression
    full_base_time_index = pd.Period(historical_data['quarter_str'].min(), freq='Q').ordinal
    category_historical['time_index'] = category_historical['quarter_str'].apply(lambda q: pd.Period(q, freq='Q').ordinal - full_base_time_index)

    X_hist = category_historical[['time_index']]
    y_hist = category_historical['incident_count']

    model = LinearRegression()
    model.fit(X_hist, y_hist)

    # Create future time points (3 years = 12 quarters)
    last_period = pd.Period(historical_data['quarter_str'].max(), freq='Q')
    future_periods = pd.period_range(start=last_period + 1, periods=12, freq='Q')
    future_time_index = [p.ordinal - full_base_time_index for p in future_periods]

    # Predict future incident counts
    future_predictions = model.predict(np.array(future_time_index).reshape(-1, 1))
    future_predictions[future_predictions < 0] = 0

    # Create the forecast DataFrame
    category_forecast = pd.DataFrame({
        'quarter_str': [str(p) for p in future_periods],
        'incident_count': future_predictions.round().astype(int),
        'type': 'Forecast'
    })

    # Combine historical and forecast data
    final_df = pd.concat([category_historical, category_forecast])

    # Drop the temporary 'time_index' column
    if 'time_index' in final_df.columns:
        final_df = final_df.drop(columns=['time_index'])

    return final_df.to_dict(orient='records')

# Get all unique categories
all_categories = df_incidents['harm_category'].unique()
all_forecasts_unordered = {}

# Use the 'complete' data (without the last partial quarter) for training the forecasts
risk_radar_complete = df_incidents_complete.groupby(['quarter_str', 'harm_category']).size().reset_index(name='incident_count')

for category in all_categories:
    print(f"-> Generating forecast for: {category}")
    all_forecasts_unordered[category] = generate_forecast_for_category(category, risk_radar_complete)

print("✅ All category forecasts are ready.")


-> Generating forecast for: System & Task Mismatch
-> Generating forecast for: Safety, Robustness & Reliability
-> Generating forecast for: Fairness, Bias & Discrimination
-> Generating forecast for: Uncategorized
-> Generating forecast for: Data Quality & Integrity
-> Generating forecast for: Malicious Use & Security
-> Generating forecast for: Human-Computer Interaction & Autonomy
-> Generating forecast for: Privacy & Data Protection
-> Generating forecast for: Societal & Economic Impact
-> Generating forecast for: Transparency & Explainability
✅ All category forecasts are ready.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/

In [58]:
# ==============================================================================
# Rank Categories and Re-order the Forecast Dictionary
# ==============================================================================
print("\n--- Ranking categories by recent growth trend ---")

category_slopes = {}
# We use the same 'complete' and recent data logic for consistency
for category in all_categories:
    category_data = risk_radar_complete[risk_radar_complete['harm_category'] == category].copy()

    # Only use the last 3 years (12 quarters) for the trend calculation
    category_data = category_data.tail(12)

    if len(category_data) > 1:
        # Create a simple, local time index just for this trend calculation
        category_data['time_index'] = range(len(category_data))

        X = category_data[['time_index']]
        y = category_data['incident_count']

        model = LinearRegression()
        model.fit(X, y)
        category_slopes[category] = model.coef_[0]
    else:
        # Assign a neutral slope if there's not enough data to calculate a trend
        category_slopes[category] = 0

# --- Get the sorted list of category names ---
sorted_categories = sorted(category_slopes, key=category_slopes.get, reverse=True)
print(f"-> Categories ranked by trend: {sorted_categories}")

# --- Build the new, ORDERED dictionary for the final JSON ---
all_forecasts_ordered = []
for category_name in sorted_categories:
    all_forecasts_ordered.append({
        "category_name": category_name,
        "data": all_forecasts_unordered[category_name]
    })

print("✅ Forecast dictionary has been re-ordered based on trend.")


--- Ranking categories by recent growth trend ---
-> Categories ranked by trend: ['Malicious Use & Security', 'Transparency & Explainability', 'Societal & Economic Impact', 'Safety, Robustness & Reliability', 'Privacy & Data Protection', 'Uncategorized', 'Fairness, Bias & Discrimination', 'System & Task Mismatch', 'Data Quality & Integrity', 'Human-Computer Interaction & Autonomy']
✅ Forecast dictionary has been re-ordered based on trend.


#Package All Outputs into a Single JSON File

In [59]:
final_output = {
    "risk_radar_data": risk_radar_data_json,
    "cause_effect_data": cause_effect_data_json,
    # The new structure for forecasts
    "all_forecasts_data": all_forecasts_ordered
}

output_filename = 'final_outputs.json'
with open(output_filename, 'w') as f:
    json.dump(final_output, f, indent=4)

print(f"✅ Successfully saved all processed data to '{output_filename}'.")

files.download(output_filename)
print(f"✅ The file '{output_filename}' has been downloaded to your computer.")

✅ Successfully saved all processed data to 'final_outputs.json'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ The file 'final_outputs.json' has been downloaded to your computer.
